In [14]:
# ============================================================
# Wavelet-space registration training
# Full GT-DVF -> low GT-DVF (*0.5) -> wavelet STN
# -> inverse Haar reconstruction
# ============================================================

import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.auto import tqdm
import voxelmorph as vxm



In [16]:
# ------------------------------------------------------------
# Training data load
# ------------------------------------------------------------

TRAIN_DATA_PATH = Path(
    r"D:\Saito\Data\TrainData_NoBed.npz"
)

if not TRAIN_DATA_PATH.exists():
    raise FileNotFoundError(
        f"学習データが見つかりません: {TRAIN_DATA_PATH}"
    )

with np.load(TRAIN_DATA_PATH) as data:
    x_raw = data["Train"]

print("Raw Train shape:", x_raw.shape)

# (D, H, W, N) -> (N, D, H, W)
if tuple(x_raw.shape[:3]) == FULL_SIZE:
    x_train = np.transpose(
        x_raw,
        (3, 0, 1, 2),
    )

# すでに (N, D, H, W) の場合
elif tuple(x_raw.shape[1:]) == FULL_SIZE:
    x_train = x_raw

else:
    raise ValueError(
        "Trainデータの形状が想定外です: "
        f"{x_raw.shape}。"
        f"期待値は (128,256,256,N) または "
        f"(N,128,256,256) です。"
    )

x_train = np.ascontiguousarray(
    x_train,
    dtype=np.float32,
)

print("x_train shape:", x_train.shape)
print("x_train range:", x_train.min(), "to", x_train.max())

Raw Train shape: (128, 256, 256, 400)
x_train shape: (400, 128, 256, 256)
x_train range: 0.0 to 1.0


In [17]:
# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

FULL_SIZE = (128, 256, 256)
LOW_SIZE = (64, 128, 128)

BATCH_SIZE = 2
LEARNING_RATE = 1e-4

PRETRAIN_EPOCHS = 80_000
FINETUNE_EPOCHS = 30_000

PRETRAIN_SAVE_PATH = Path(
    "pretrained_wavelet_gt_dvf.pth"
)

FINETUNE_SAVE_PATH = Path(
    "finetuned_wavelet_gt_dvf.pth"
)

# x_train は (N, 128, 256, 256) を想定
assert x_train.ndim == 4, x_train.shape
assert tuple(x_train.shape[1:]) == FULL_SIZE, x_train.shape



In [18]:
# ------------------------------------------------------------
# Data generator
# ------------------------------------------------------------

def data_generator(data, batch_size):
    while True:
        moving_index = np.random.randint(
            0,
            len(data),
            size=batch_size,
        )

        fixed_index = np.random.randint(
            0,
            len(data),
            size=batch_size,
        )

        # Fine-tuningで同一画像ペアを避ける
        while np.any(moving_index == fixed_index):
            same = moving_index == fixed_index
            fixed_index[same] = np.random.randint(
                0,
                len(data),
                size=int(same.sum()),
            )

        moving = torch.from_numpy(
            data[moving_index]
        ).unsqueeze(1).float()

        fixed = torch.from_numpy(
            data[fixed_index]
        ).unsqueeze(1).float()

        yield moving.to(device), fixed.to(device)


train_generator = data_generator(
    x_train.astype(np.float32),
    BATCH_SIZE,
)


In [19]:

# ------------------------------------------------------------
# Haar analysis / inverse Haar synthesis
# ------------------------------------------------------------

def make_haar_filters():
    h_low = torch.tensor(
        [1.0, 1.0],
        dtype=torch.float32,
    ) / math.sqrt(2.0)

    h_high = torch.tensor(
        [1.0, -1.0],
        dtype=torch.float32,
    ) / math.sqrt(2.0)

    filters = []

    for z_filter in [h_low, h_high]:
        for y_filter in [h_low, h_high]:
            for x_filter in [h_low, h_high]:
                kernel = (
                    z_filter[:, None, None]
                    * y_filter[None, :, None]
                    * x_filter[None, None, :]
                )
                filters.append(kernel)

    # (8, 1, 2, 2, 2)
    return torch.stack(filters, dim=0).unsqueeze(1)


HAAR_FILTERS = make_haar_filters().to(device)


def haar_wavelet_3d(image):
    """
    image: (B, 1, 128, 256, 256)
    return: (B, 8, 64, 128, 128)
    """
    filters = HAAR_FILTERS.to(
        device=image.device,
        dtype=image.dtype,
    )

    return F.conv3d(
        image,
        filters,
        stride=2,
        padding=0,
    )


def inverse_haar_wavelet_3d(coefficients):
    """
    coefficients: (B, 8, 64, 128, 128)
    return      : (B, 1, 128, 256, 256)
    """
    filters = HAAR_FILTERS.to(
        device=coefficients.device,
        dtype=coefficients.dtype,
    )

    return F.conv_transpose3d(
        coefficients,
        filters,
        stride=2,
        padding=0,
    )

In [20]:

# ------------------------------------------------------------
# Spatial transformers
# ------------------------------------------------------------

transformer_low = vxm.layers.SpatialTransformer(
    LOW_SIZE
).to(device)

transformer_full = vxm.layers.SpatialTransformer(
    FULL_SIZE
).to(device)


In [21]:

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16],
]

model3D = vxm.networks.VxmDense_128_256_256(
    FULL_SIZE,
    nb_features,
    int_steps=0,
).to(device)

optimizer = optim.Adam(
    model3D.parameters(),
    lr=LEARNING_RATE,
)

[64, 128, 128]


In [22]:

# ------------------------------------------------------------
# DVF utilities
# ------------------------------------------------------------

def gaussian_smooth_3d(flow, sigma=2.0):
    """
    flow: (B, 3, D, H, W)
    """
    radius = max(1, int(math.ceil(3.0 * sigma)))

    coordinate = torch.arange(
        -radius,
        radius + 1,
        device=flow.device,
        dtype=flow.dtype,
    )

    kernel_1d = torch.exp(
        -(coordinate ** 2) / (2.0 * sigma ** 2)
    )
    kernel_1d = kernel_1d / kernel_1d.sum()

    channels = flow.shape[1]

    kernel_z = kernel_1d.view(1, 1, -1, 1, 1).repeat(
        channels, 1, 1, 1, 1
    )
    kernel_y = kernel_1d.view(1, 1, 1, -1, 1).repeat(
        channels, 1, 1, 1, 1
    )
    kernel_x = kernel_1d.view(1, 1, 1, 1, -1).repeat(
        channels, 1, 1, 1, 1
    )

    flow = F.conv3d(
        flow,
        kernel_z,
        padding=(radius, 0, 0),
        groups=channels,
    )
    flow = F.conv3d(
        flow,
        kernel_y,
        padding=(0, radius, 0),
        groups=channels,
    )
    flow = F.conv3d(
        flow,
        kernel_x,
        padding=(0, 0, radius),
        groups=channels,
    )

    return flow


def make_full_gt_dvf(
    batch_size,
    shift_range,
):
    """
    Full-resolution voxel単位の正解DVFを作る。
    """
    coarse_flow = (
        torch.rand(
            batch_size,
            3,
            8,
            16,
            16,
            device=device,
        ) * 2.0 - 1.0
    ) * shift_range

    coarse_flow = gaussian_smooth_3d(
        coarse_flow,
        sigma=2.0,
    )

    return F.interpolate(
        coarse_flow,
        size=FULL_SIZE,
        mode="trilinear",
        align_corners=False,
    )


def full_dvf_to_low_dvf(full_dvf):
    """
    Full-res  : (B, 3, 128, 256, 256)
    Low-res   : (B, 3,  64, 128, 128)

    座標間隔が2倍になるため、DVFのvoxel移動量は半分にする。
    """
    return F.interpolate(
        full_dvf,
        size=LOW_SIZE,
        mode="trilinear",
        align_corners=False,
    ) * 0.5


def warp_all_wavelet_bands(
    wavelet_image,
    dvf_low,
):
    """
    wavelet_image: (B, 8, 64, 128, 128)
    dvf_low      : (B, 3, 64, 128, 128)
    """
    warped_bands = []

    for channel in range(8):
        warped_band = transformer_low(
            wavelet_image[:, channel:channel + 1],
            dvf_low,
        )
        warped_bands.append(warped_band)

    return torch.cat(
        warped_bands,
        dim=1,
    )


In [ ]:

# ============================================================
# Pretraining
# 正解full-DVFを low-DVFへ縮小してwavelet STNで教師画像を作る
# ============================================================

model3D.train()

for epoch in tqdm(
    range(1, PRETRAIN_EPOCHS + 1),
    desc="Pretraining",
):
    moving_images, _ = next(train_generator)

    # 徐々に大きな変位を学習
    shift_range = 1 + (epoch - 1) // 2000

    # 1. full-resolution 正解DVF
    gt_dvf_full = make_full_gt_dvf(
        batch_size=moving_images.shape[0],
        shift_range=shift_range,
    )

    # 2. low-resolution用の正解DVF
    #    空間サイズ1/2、移動量も1/2
    gt_dvf_low = full_dvf_to_low_dvf(
        gt_dvf_full
    )

    # 3. Movingをwavelet化
    moving_w = haar_wavelet_3d(
        moving_images
    )

    # 4. 正解low-DVFをwavelet STNへ適用
    target_w = warp_all_wavelet_bands(
        moving_w,
        gt_dvf_low,
    )

    # 5. 逆wavelet変換して教師画像を再構成
    target_image = inverse_haar_wavelet_3d(
        target_w
    )

    # 6. 教師画像もwavelet化してモデル入力にする
    target_w_input = haar_wavelet_3d(
        target_image
    )

    # 7. 低解像度DVFを予測
    optimizer.zero_grad()

    pred_dvf_low = model3D(
        moving_w,
        target_w_input,
    )

    # 8. 予測DVFでMovingをwavelet空間で変形・再構成
    pred_w = warp_all_wavelet_bands(
        moving_w,
        pred_dvf_low,
    )

    pred_image = inverse_haar_wavelet_3d(
        pred_w
    )

    # 9. 損失
    loss_dvf = F.mse_loss(
        pred_dvf_low,
        gt_dvf_low,
    )

    loss_image = F.mse_loss(
        pred_image,
        target_image,
    )

    loss = 0.01 * loss_dvf + 100.0 * loss_image

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"epoch={epoch:05d} | "
            f"loss={loss.item():.6f} | "
            f"dvf={loss_dvf.item():.6f} | "
            f"image={loss_image.item():.6f} | "
            f"shift=±{shift_range}"
        )

    if epoch % 1000 == 0:
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model3D.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            PRETRAIN_SAVE_PATH,
        )



Pretraining:   0%|          | 0/80000 [00:00<?, ?it/s]

KeyboardInterrupt: 

: 

In [ ]:
# ============================================================
# Fine-tuning
# 実際の Moving / Fixed ペアで画像MSEのみ最適化
# ============================================================

model3D.train()

for epoch in tqdm(
    range(1, FINETUNE_EPOCHS + 1),
    desc="Fine-tuning",
):
    moving_images, fixed_images = next(train_generator)

    moving_w = haar_wavelet_3d(
        moving_images
    )
    fixed_w = haar_wavelet_3d(
        fixed_images
    )

    optimizer.zero_grad()

    pred_dvf_low = model3D(
        moving_w,
        fixed_w,
    )

    pred_w = warp_all_wavelet_bands(
        moving_w,
        pred_dvf_low,
    )

    pred_image = inverse_haar_wavelet_3d(
        pred_w
    )

    loss = F.mse_loss(
        pred_image,
        fixed_images,
    )

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"fine-tune={epoch:05d} | "
            f"loss={loss.item():.6f}"
        )

    if epoch % 1000 == 0:
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model3D.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            FINETUNE_SAVE_PATH,
        )


torch.save(
    {
        "epoch": FINETUNE_EPOCHS,
        "model_state_dict": model3D.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    },
    FINETUNE_SAVE_PATH,
)

print("Training complete.")
print("Saved:", FINETUNE_SAVE_PATH)

In [ ]:
import matplotlib.pyplot as plt

z = FULL_SIZE[0] // 2

moving_np = moving_images[0, 0, z].detach().cpu().numpy()
target_np = target_image[0, 0, z].detach().cpu().numpy()
pred_np = pred_image[0, 0, z].detach().cpu().numpy()

vmin, vmax = np.percentile(
    np.concatenate([
        moving_np.ravel(),
        target_np.ravel(),
        pred_np.ravel(),
    ]),
    [1, 99],
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
    constrained_layout=True,
)

for axis, image, title in zip(
    axes,
    [moving_np, target_np, pred_np],
    ["Moving", "GT-DVF warped target", "Predicted-DVF warped"],
):
    axis.imshow(
        image,
        cmap="gray",
        vmin=vmin,
        vmax=vmax,
    )
    axis.set_title(title)
    axis.axis("off")

fig.suptitle(
    f"Pretraining epoch {epoch} | "
    f"loss={loss.item():.6f}"
)

plt.show()
plt.close(fig)